# Sensex Weekly Expiry — Hourly Move Table From A Session-2 11:15 Entry

A variant of `weekly_expiry_hourly_move_table.ipynb`. Instead of anchoring each week on the previous expiry's close, every week is measured from an **entry on the 2nd trading session after the previous expiry, at the open of the 11:15 hourly candle**:

- Friday expiry (May 2023 – Dec 2024) → entry Tuesday 11:15
- Tuesday expiry (Jan – Aug 2025) → entry Thursday 11:15
- Thursday expiry (Sept 2025 onwards) → entry Monday 11:15

When a holiday falls in between, the entry moves one session later.

Sensex weekly options have changed expiry day twice: **Friday** from their May 2023 launch, **Tuesday** from Jan 2025 and **Thursday** from Sept 2025. Before May 2023 only a monthly contract listed; those monthly-only cycles are skipped (see `MAX_EXPIRY_GAP_DAYS`), so the weekly table covers May 2023 onwards.

Holidays are handled by counting *regular* traded sessions, so the entry always lands on the 2nd weekday session that actually traded; weekend special sessions and Muhurat prints are not counted. From that entry this notebook measures, up to the next expiry:
- the max upside % and max downside % reached at any point,
- the hour at which each extreme printed,
- the net move from the entry price to the close at expiry,
- and how often a strike that was *touched* still expired worthless.

Source data:
- `data/raw/option_selling/sensex/sensex_expiry_dates.csv` — scheduled and actual expiry dates (monthly-only before May 2023, weekly after)
- `data/raw/option_selling/sensex/sensex_hourly_2020-01-01_2026-09-19.json` — hourly OHLC candles (data starts 2022-01-03)

Change `ENTRY_SESSION_OFFSET` or `ENTRY_TIME` in the parameters cell to test a different entry; `SKIP_FIRST_CANDLES` drops the first N hourly candles after the entry.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    """Walk up from the notebook's location until the folder holding data/raw is found."""
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/raw")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "option_selling" / "sensex"

EXPIRY_CSV = RAW_DIR / "sensex_expiry_dates.csv"
CANDLES_JSON = RAW_DIR / "sensex_hourly_2020-01-01_2026-09-19.json"

# --- Parameters -------------------------------------------------------------
ENTRY_SESSION_OFFSET = 2                                     # enter on the Nth traded session after the prior expiry
ENTRY_TIME = "11:15"                                         # entry candle start (IST); its open is the entry price
SKIP_FIRST_CANDLES = 0                                       # drop the first N hourly candles after the entry
MAX_EXPIRY_GAP_DAYS = 12                                     # longer expiry-to-expiry cycles are monthly-only and skipped
STRIKE_DISTANCES = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]  # % away from the entry price
TOUCH_THRESHOLDS = [1.0, 2.0, 3.0]                           # thresholds tracked in the intraweek timing curve
REGULAR_SESSION_END_HOUR = 16                                # candles stamped later are Muhurat/special sessions
REGULAR_SESSION_MIN_CANDLES = 6                              # weekday sessions with fewer candles (Muhurat) are not trading days
BIN_WIDTH, LOWER_TAIL, UPPER_TAIL = 1.0, -8.0, 6.0           # return-distribution buckets
# ---------------------------------------------------------------------------

ENTRY_LABEL = f"{ENTRY_TIME} Entry On Session {ENTRY_SESSION_OFFSET} After Expiry"

# Diverging poles with a neutral midpoint: upside = warm, downside = cool.
UP_COLOR = (235, 104, 52)
DOWN_COLOR = (42, 120, 214)
NEUTRAL = (240, 239, 236)
UP_HEX, DOWN_HEX = "#eb6834", "#2a78d6"
SERIES_HEX = ["#2a78d6", "#eb6834", "#1baf7a"]   # fixed categorical order, never cycled
HEADER_FILL = "#2b2d42"

pd.options.display.float_format = "{:,.2f}".format

## Load Hourly Candles & Weekly Expiries

Expiry dates come from the exchange calendar, candles from the hourly dump. These clean-ups happen here:

- **Muhurat / special sessions** (single candles stamped after 16:00) are dropped — they are not part of a normal expiry week.
- **Regular sessions** are weekdays with at least `REGULAR_SESSION_MIN_CANDLES` hourly candles. Weekend special sessions (Budget days, exchange DR drills) and one-candle Muhurat prints are not counted as trading days — they never become an expiry or an entry day — but their candles stay inside any window that spans them.
- **One expiry per week.** The calendar also lists a last-Tuesday expiry in Sept–Dec 2025, two days before that week's Thursday weekly. When a Mon–Sun week holds two expiries only the later one is kept, so the chain follows the weekly contract instead of splitting the week into 1–3 session stubs.
- Each expiry date is **snapped back to the last regular session on or before it**, so a holiday-shifted expiry still settles on a real closing print. Duplicates after snapping are removed.
- **Weekly cycles only.** Consecutive expiries more than `MAX_EXPIRY_GAP_DAYS` calendar days apart are monthly-only cycles (Sensex before its May 2023 weekly launch) and are skipped when the windows are built. The 11-day Friday → Tuesday switchover week at the turn of 2025 is kept.

In [2]:
expiry_df = pd.read_csv(EXPIRY_CSV, parse_dates=["Actual_Expiry_Date"])
expiry_dates = expiry_df["Actual_Expiry_Date"].dt.normalize().sort_values().reset_index(drop=True)
earlier_in_week = expiry_dates.dt.to_period("W").duplicated(keep="last")   # off-cycle expiry sharing a Mon–Sun week
off_cycle = expiry_dates[earlier_in_week]
expiry_dates = expiry_dates[~earlier_in_week].reset_index(drop=True)

with open(CANDLES_JSON) as f:
    raw = json.load(f)

price_df = pd.DataFrame(
    raw["data"]["candles"],
    columns=["timestamp", "open", "high", "low", "close", "volume", "oi"],
)
price_df["timestamp"] = (
    pd.to_datetime(price_df["timestamp"], utc=True).dt.tz_convert("Asia/Kolkata").dt.tz_localize(None)
)
price_df = price_df.drop_duplicates("timestamp").set_index("timestamp").sort_index()
price_df = price_df[["open", "high", "low", "close"]].apply(pd.to_numeric)
price_df = price_df[price_df.index.hour < REGULAR_SESSION_END_HOUR]
price_df["session"] = price_df.index.normalize()

sessions = pd.DatetimeIndex(price_df["session"].unique())
candles_per_session = price_df.groupby("session").size()
regular_sessions = candles_per_session.index[
    (candles_per_session.index.dayofweek < 5) & (candles_per_session.to_numpy() >= REGULAR_SESSION_MIN_CANDLES)
]

in_range = expiry_dates[(expiry_dates >= sessions.min()) & (expiry_dates <= sessions.max())]
snapped = pd.Series(regular_sessions[regular_sessions.searchsorted(in_range, side="right") - 1])
expiry_sessions = snapped.drop_duplicates().reset_index(drop=True)
shifted_count = int((snapped.to_numpy() != in_range.to_numpy()).sum())
long_cycles = expiry_sessions[expiry_sessions.diff().dt.days > MAX_EXPIRY_GAP_DAYS]

print(
    f"Loaded {len(price_df):,} hourly candles over {len(sessions):,} sessions, "
    f"{price_df.index.min():%Y-%m-%d %H:%M} to {price_df.index.max():%Y-%m-%d %H:%M}"
)
print(f"Candles per session: {price_df.groupby('session').size().median():.0f} (median)")
print(f"{len(expiry_sessions)} expiries in range, {shifted_count} snapped back to the previous regular session")
print(f"{len(off_cycle)} off-cycle expiry date(s) dropped — an earlier second expiry in the same week: "
      + ", ".join(f"{d:%Y-%m-%d %a}" for d in off_cycle))
print(f"{len(long_cycles)} expiry cycle(s) longer than {MAX_EXPIRY_GAP_DAYS} days skipped as monthly-only, "
      f"leaving {len(expiry_sessions) - 1 - len(long_cycles)} weekly cycles. Skipped cycles end on: "
      + ", ".join(f"{d:%Y-%m-%d}" for d in long_cycles))
irregular = sessions.difference(regular_sessions)
print(f"{len(irregular)} non-regular session(s) excluded from expiry/entry days: "
      + ", ".join(f"{d:%Y-%m-%d %a} ({candles_per_session[d]} candles)" for d in irregular))
display(price_df.head())

Loaded 8,149 hourly candles over 1,167 sessions, 2022-01-03 09:15 to 2026-09-18 15:15
Candles per session: 7 (median)
190 expiries in range, 1 snapped back to the previous regular session
4 off-cycle expiry date(s) dropped — an earlier second expiry in the same week: 2025-09-30 Tue, 2025-10-28 Tue, 2025-11-25 Tue, 2025-12-30 Tue
16 expiry cycle(s) longer than 12 days skipped as monthly-only, leaving 173 weekly cycles. Skipped cycles end on: 2022-02-24, 2022-03-31, 2022-04-28, 2022-05-26, 2022-06-30, 2022-07-28, 2022-08-25, 2022-09-29, 2022-10-27, 2022-11-24, 2022-12-29, 2023-01-25, 2023-02-23, 2023-03-29, 2023-04-27, 2023-05-19
7 non-regular session(s) excluded from expiry/entry days: 2022-08-09 Tue (1 candles), 2024-01-20 Sat (7 candles), 2024-03-02 Sat (3 candles), 2024-05-18 Sat (3 candles), 2025-02-01 Sat (7 candles), 2025-10-21 Tue (1 candles), 2026-02-01 Sun (7 candles)


,open,high,low,close,session
timestamp,,,,,
2022-01-03 09:15:00,"58,310.09","58,773.67","58,310.09","58,758.90",2022-01-03
2022-01-03 10:15:00,"58,760.98","58,858.35","58,721.51","58,855.62",2022-01-03
2022-01-03 11:15:00,"58,856.28","58,875.67","58,804.98","58,850.66",2022-01-03
2022-01-03 12:15:00,"58,849.36","59,037.63","58,844.62","59,027.76",2022-01-03
2022-01-03 13:15:00,"59,027.85","59,154.98","58,998.40","59,141.63",2022-01-03


## Build Each Entry-To-Expiry Window

For each pair of consecutive expiries, the **entry candle** is the first hourly candle at or after `ENTRY_TIME` on the `ENTRY_SESSION_OFFSET`-th regular session after the previous expiry. The window runs from that candle (inclusive — its own high and low count) through the last candle of the next expiry session. A week is skipped if the entry session would fall on or after the next expiry, and pairs of expiries more than `MAX_EXPIRY_GAP_DAYS` apart are monthly-only cycles and are skipped too.

- **entry_price** — open of the entry candle; every % below is measured against it
- **max_up_pct** — highest hourly high in the window vs `entry_price`
- **max_down_pct** — lowest hourly low in the window vs `entry_price`
- **expiry_move_pct** — entry price to the close of the expiry session
- **candles_to_max_up / down** — how many hourly candles after entry the extreme printed
- **session_of_max_up / down** — which session of the window it printed in (1 = entry day)

`build_windows` also returns the hourly *path* of every week (running % vs the entry price), which the intraweek timing section consumes.

In [3]:
def entry_candle(price: pd.DataFrame, prev_expiry: pd.Timestamp, curr_expiry: pd.Timestamp):
    """First candle at or after ENTRY_TIME on the ENTRY_SESSION_OFFSET-th regular session after `prev_expiry`.

    Returns None when that session does not exist or is not strictly before the next expiry.
    """
    pos = regular_sessions.get_loc(prev_expiry) + ENTRY_SESSION_OFFSET
    if pos >= len(regular_sessions) or regular_sessions[pos] >= curr_expiry:
        return None
    day = price[price["session"] == regular_sessions[pos]]
    at_or_after = day.index[day.index.strftime("%H:%M") >= ENTRY_TIME]
    return at_or_after[0] if len(at_or_after) else None


def build_windows(price: pd.DataFrame, expiries: pd.Series, skip_first: int = 0):
    """Measure every entry-to-expiry window on hourly candles, anchored on the entry candle's open.

    Returns the per-week summary table, for each week the hourly (up %, down %) path relative to
    the entry price, and the expiries whose week had no valid entry.
    """
    rows, paths, skipped = [], [], []
    for prev_expiry, curr_expiry in zip(expiries.iloc[:-1], expiries.iloc[1:]):
        if (curr_expiry - prev_expiry).days > MAX_EXPIRY_GAP_DAYS:
            continue   # monthly-only cycle, not an expiry week
        entry_ts = entry_candle(price, prev_expiry, curr_expiry)
        if entry_ts is None:
            skipped.append(curr_expiry)
            continue
        window = price[(price.index >= entry_ts) & (price["session"] <= curr_expiry)]
        if skip_first:
            window = window.iloc[skip_first:]
        if window.empty:
            skipped.append(curr_expiry)
            continue

        entry_price = price.at[entry_ts, "open"]
        up_path = (window["high"] - entry_price) / entry_price * 100
        down_path = (window["low"] - entry_price) / entry_price * 100
        up_ts, down_ts = window["high"].idxmax(), window["low"].idxmin()
        expiry_close = window["close"].iloc[-1]
        session_no = window["session"].rank(method="dense").astype(int)   # 1 = first session in the window

        rows.append(
            {
                "prev_expiry": prev_expiry,
                "curr_expiry": curr_expiry,
                "entry_ts": entry_ts,
                "entry_price": entry_price,
                "max_up_pct": up_path.max(),
                "max_up_ts": up_ts,
                "candles_to_max_up": window.index.get_loc(up_ts) + 1,
                "session_of_max_up": int(session_no.at[up_ts]),
                "max_down_pct": down_path.min(),
                "max_down_ts": down_ts,
                "candles_to_max_down": window.index.get_loc(down_ts) + 1,
                "session_of_max_down": int(session_no.at[down_ts]),
                "expiry_close": expiry_close,
                "expiry_move_pct": (expiry_close - entry_price) / entry_price * 100,
                "candles": len(window),
                "first_session_candles": int((session_no == 1).sum()),
                "trading_days": window["session"].nunique(),
            }
        )
        paths.append(np.column_stack([up_path.to_numpy(), down_path.to_numpy()]))

    return pd.DataFrame(rows), paths, skipped


moves, paths, skipped_weeks = build_windows(price_df, expiry_sessions, SKIP_FIRST_CANDLES)
moves["year"] = moves["curr_expiry"].dt.year
moves["range_pct"] = moves["max_up_pct"] - moves["max_down_pct"]

late_entries = moves[moves["entry_ts"].dt.strftime("%H:%M") != ENTRY_TIME]
print(f"{len(moves)} entry-to-expiry weeks computed (skip_first_candles={SKIP_FIRST_CANDLES})")
print("Entry weekday:", moves["entry_ts"].dt.day_name().value_counts().to_dict())
print(f"{len(late_entries)} week(s) had no {ENTRY_TIME} candle and entered on the next candle that session")
if skipped_weeks:
    print(f"{len(skipped_weeks)} week(s) skipped — no session {ENTRY_SESSION_OFFSET} before expiry: "
          + ", ".join(f"{d:%Y-%m-%d}" for d in skipped_weeks))
display(moves.tail(10))

173 entry-to-expiry weeks computed (skip_first_candles=0)
Entry weekday: {'Tuesday': 79, 'Monday': 49, 'Thursday': 30, 'Wednesday': 11, 'Friday': 4}
0 week(s) had no 11:15 candle and entered on the next candle that session


,prev_expiry,curr_expiry,entry_ts,entry_price,max_up_pct,max_up_ts,candles_to_max_up,session_of_max_up,max_down_pct,max_down_ts,candles_to_max_down,session_of_max_down,expiry_close,expiry_move_pct,candles,first_session_candles,trading_days,year,range_pct
163,2026-07-09,2026-07-16,2026-07-13 11:15:00,"77,399.14",0.50,2026-07-13 12:15:00,2,1,-0.54,2026-07-15 12:15:00,16,3,"77,258.32",-0.18,26,5,4,2026,1.04
164,2026-07-16,2026-07-23,2026-07-20 11:15:00,"77,653.83",0.21,2026-07-20 13:15:00,3,1,-1.93,2026-07-23 14:15:00,25,4,"76,377.94",-1.64,26,5,4,2026,2.14
165,2026-07-23,2026-07-30,2026-07-27 11:15:00,"76,641.15",1.78,2026-07-30 15:15:00,26,4,-0.01,2026-07-27 11:15:00,1,1,"77,835.43",1.56,26,5,4,2026,1.79
166,2026-07-30,2026-08-06,2026-08-03 11:15:00,"78,737.35",0.52,2026-08-04 09:15:00,6,2,-0.67,2026-08-04 13:15:00,10,2,"78,954.76",0.28,26,5,4,2026,1.18
167,2026-08-06,2026-08-13,2026-08-10 11:15:00,"78,602.69",0.08,2026-08-10 12:15:00,2,1,-1.41,2026-08-12 11:15:00,15,3,"78,079.96",-0.67,26,5,4,2026,1.49
168,2026-08-13,2026-08-20,2026-08-17 11:15:00,"77,486.06",0.56,2026-08-17 14:15:00,4,1,-0.86,2026-08-19 14:15:00,18,3,"77,537.72",0.07,26,5,4,2026,1.41
169,2026-08-20,2026-08-27,2026-08-24 11:15:00,"77,415.42",0.74,2026-08-26 09:15:00,13,3,-0.62,2026-08-27 15:15:00,26,4,"76,933.59",-0.62,26,5,4,2026,1.36
170,2026-08-27,2026-09-03,2026-08-31 11:15:00,"76,787.66",0.58,2026-09-01 11:15:00,8,2,-0.85,2026-09-02 09:15:00,13,3,"76,152.86",-0.83,26,5,4,2026,1.43
171,2026-09-03,2026-09-10,2026-09-07 11:15:00,"76,098.62",0.05,2026-09-07 11:15:00,1,1,-1.97,2026-09-10 14:15:00,25,4,"74,902.59",-1.57,26,5,4,2026,2.02
172,2026-09-10,2026-09-17,2026-09-15 11:15:00,"74,667.76",0.08,2026-09-15 11:15:00,1,1,-0.92,2026-09-16 09:15:00,6,2,"74,314.59",-0.47,19,5,3,2026,1.00


## Summary Stats

In [4]:
summary = pd.DataFrame(
    [
        ("Weeks analyzed", f"{len(moves)}"),
        ("Hourly candles per week (median)", f"{moves['candles'].median():.0f}"),
        ("Avg max up %", f"{moves['max_up_pct'].mean():.2f}%"),
        ("Avg max down %", f"{moves['max_down_pct'].mean():.2f}%"),
        ("Avg intraweek range (up − down)", f"{moves['range_pct'].mean():.2f}%"),
        (
            "Biggest single-week rally",
            f"{moves['max_up_pct'].max():.2f}% (expiry {moves.loc[moves['max_up_pct'].idxmax(), 'curr_expiry'].date()})",
        ),
        (
            "Biggest single-week crash",
            f"{moves['max_down_pct'].min():.2f}% (expiry {moves.loc[moves['max_down_pct'].idxmin(), 'curr_expiry'].date()})",
        ),
        ("Avg entry-to-expiry close move", f"{moves['expiry_move_pct'].mean():.2f}%"),
        ("Weeks closed higher", f"{(moves['expiry_move_pct'] > 0).sum()} / {len(moves)}"),
        ("Weeks closed lower", f"{(moves['expiry_move_pct'] < 0).sum()} / {len(moves)}"),
        ("Median candle (after entry) of the weekly high", f"{moves['candles_to_max_up'].median():.0f}"),
        ("Median candle (after entry) of the weekly low", f"{moves['candles_to_max_down'].median():.0f}"),
    ],
    columns=["Metric", "Value"],
)
display(summary)

,Metric,Value
0,Weeks analyzed,173
1,Hourly candles per week (median),26
2,Avg max up %,1.03%
3,Avg max down %,-1.01%
4,Avg intraweek range (up − down),2.04%
5,Biggest single-week rally,6.61% (expiry 2026-02-05)
6,Biggest single-week crash,-6.51% (expiry 2025-04-08)
7,Avg entry-to-expiry close move,0.09%
8,Weeks closed higher,93 / 173
9,Weeks closed lower,80 / 173


## Visual Table — Max Up % / Max Down % Per Expiry Week

Most recent week first. Shading encodes magnitude — deeper warm for a larger max upside, deeper cool for a larger max downside — and the signed value is printed in every cell, so direction never rests on colour alone. The *At* columns carry the hourly timestamp of each extreme.

In [5]:
def shade(values: pd.Series, color: tuple[int, int, int]) -> list[str]:
    """Blend from the neutral surface toward `color` in proportion to each value's magnitude."""
    magnitude = values.abs()
    peak = magnitude.max() or 1.0
    intensity = (magnitude / peak).clip(0, 1)
    return [
        "rgb({},{},{})".format(*[int(NEUTRAL[i] + (color[i] - NEUTRAL[i]) * t) for i in range(3)])
        for t in intensity
    ]


table_df = moves.sort_values("curr_expiry", ascending=False).reset_index(drop=True)
white = ["white"] * len(table_df)
close_colors = [
    f"rgb{UP_COLOR}" if v > 0 else (f"rgb{DOWN_COLOR}" if v < 0 else f"rgb{NEUTRAL}")
    for v in table_df["expiry_move_pct"]
]

fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[120, 85, 80, 75, 120, 80, 120, 85],
            header=dict(
                values=[
                    "Entry At", "Expiry", "Entry Price", "Max Up %", "Max Up At",
                    "Max Down %", "Max Down At", "Expiry Move %",
                ],
                fill_color=HEADER_FILL,
                font=dict(color="white", size=12),
                align="center",
                height=32,
            ),
            cells=dict(
                values=[
                    table_df["entry_ts"].dt.strftime("%Y-%m-%d %H:%M"),
                    table_df["curr_expiry"].dt.strftime("%Y-%m-%d"),
                    table_df["entry_price"].map("{:,.2f}".format),
                    table_df["max_up_pct"].map("{:+.2f}%".format),
                    table_df["max_up_ts"].dt.strftime("%Y-%m-%d %H:%M"),
                    table_df["max_down_pct"].map("{:+.2f}%".format),
                    table_df["max_down_ts"].dt.strftime("%Y-%m-%d %H:%M"),
                    table_df["expiry_move_pct"].map("{:+.2f}%".format),
                ],
                fill_color=[
                    white, white, white,
                    shade(table_df["max_up_pct"], UP_COLOR),
                    white,
                    shade(table_df["max_down_pct"], DOWN_COLOR),
                    white,
                    close_colors,
                ],
                align="center",
                height=26,
                font=dict(size=11),
            ),
        )
    ]
)
fig.update_layout(
    title=f"Sensex — Hourly Max Up / Max Down % From The {ENTRY_LABEL}",
    height=900,
    margin=dict(t=50, b=10, l=10, r=10),
)
fig.show()

## Touched vs Settled — The Option Seller's Ladder

For each distance from the entry price, two very different probabilities:

- **Touched** — the index traded through that level at *some* hour during the week. This is what stops out a delta-hedged or stop-managed short.
- **Settled beyond** — the index was still past that level at expiry. This is what actually pays out on a held short.

The gap between the two columns is the premium a seller keeps for sitting through the noise; it is also the share of weeks where a stop-loss would have paid out on a strike that ultimately expired worthless. Hourly candles are what make the *touched* column trustworthy.

In [6]:
n_weeks = len(moves)
ladder = pd.DataFrame({"distance_pct": STRIKE_DISTANCES})
ladder["touch_up"] = [(moves["max_up_pct"] >= d).mean() * 100 for d in STRIKE_DISTANCES]
ladder["touch_down"] = [(moves["max_down_pct"] <= -d).mean() * 100 for d in STRIKE_DISTANCES]
ladder["settle_above"] = [(moves["expiry_move_pct"] >= d).mean() * 100 for d in STRIKE_DISTANCES]
ladder["settle_below"] = [(moves["expiry_move_pct"] <= -d).mean() * 100 for d in STRIKE_DISTANCES]
ladder["touched_either"] = [
    ((moves["max_up_pct"] >= d) | (moves["max_down_pct"] <= -d)).mean() * 100 for d in STRIKE_DISTANCES
]
ladder["settled_outside"] = [(moves["expiry_move_pct"].abs() >= d).mean() * 100 for d in STRIKE_DISTANCES]
ladder["strangle_survived"] = 100 - ladder["settled_outside"]

fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[80, 80, 80, 85, 85, 95, 95, 105],
            header=dict(
                values=[
                    "Distance", "Touched<br>Up", "Touched<br>Down", "Settled<br>Above",
                    "Settled<br>Below", "Touched<br>Either Side", "Settled<br>Outside",
                    "Short Strangle<br>Expired Worthless",
                ],
                fill_color=HEADER_FILL,
                font=dict(color="white", size=11),
                align="center",
                height=44,
            ),
            cells=dict(
                values=[
                    ladder["distance_pct"].map("±{:.1f}%".format),
                    ladder["touch_up"].map("{:.1f}%".format),
                    ladder["touch_down"].map("{:.1f}%".format),
                    ladder["settle_above"].map("{:.1f}%".format),
                    ladder["settle_below"].map("{:.1f}%".format),
                    ladder["touched_either"].map("{:.1f}%".format),
                    ladder["settled_outside"].map("{:.1f}%".format),
                    ladder["strangle_survived"].map("{:.1f}%".format),
                ],
                fill_color=[
                    ["#f4f3f0"] * len(ladder),
                    shade(ladder["touch_up"], UP_COLOR),
                    shade(ladder["touch_down"], DOWN_COLOR),
                    shade(ladder["settle_above"], UP_COLOR),
                    shade(ladder["settle_below"], DOWN_COLOR),
                    ["white"] * len(ladder),
                    ["white"] * len(ladder),
                    ["white"] * len(ladder),
                ],
                align="center",
                height=28,
                font=dict(size=11),
            ),
        )
    ]
)
fig.update_layout(
    title=f"Touch vs Settle Probability By Distance From The Entry Price ({n_weeks} weeks)",
    height=120 + 28 * len(ladder) + 60,
    margin=dict(t=60, b=10, l=10, r=10),
)
fig.show()

bars = go.Figure()
bars.add_trace(
    go.Bar(
        x=ladder["distance_pct"], y=ladder["touched_either"], name="Touched either side",
        marker_color=SERIES_HEX[0], marker_line=dict(color="white", width=2),
        text=ladder["touched_either"].map("{:.0f}%".format), textposition="outside",
    )
)
bars.add_trace(
    go.Bar(
        x=ladder["distance_pct"], y=ladder["settled_outside"], name="Settled outside at expiry",
        marker_color=SERIES_HEX[1], marker_line=dict(color="white", width=2),
        text=ladder["settled_outside"].map("{:.0f}%".format), textposition="outside",
    )
)
bars.update_layout(
    title="Path Risk vs Settlement Risk — How Often A Strike Is Touched But Not Breached At Expiry",
    barmode="group",
    bargap=0.25,
    template="plotly_white",
    height=440,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(t=80, b=50, l=60, r=20),
)
bars.update_xaxes(title_text="Strike distance from entry price", ticksuffix="%", tickvals=STRIKE_DISTANCES)
bars.update_yaxes(title_text="% of weeks", ticksuffix="%", range=[0, 105])
bars.show()

## When In The Week Does The Damage Happen?

Each weekly path is walked hour by hour, tracking the share of weeks that had already traded ±1%, ±2% or ±3% away from the entry price by that point. A curve that is already flat by the middle of the week means the risk was front-loaded; one that keeps climbing into expiry means the last sessions are where shorts get tested.

Dotted verticals mark session boundaries: the entry session contributes the candles from the entry onward (5 for an 11:15 entry), every later full session 7. The table below counts sessions directly, so it stays correct around holidays.

In [7]:
CANDLES_PER_SESSION = 7
first_session = int(moves["first_session_candles"].median())
max_candles = max(len(p) for p in paths)

# Running |move| from the entry price: how far the week had travelled by candle k, either side.
excursion = np.full((len(paths), max_candles), np.nan)
for i, path in enumerate(paths):
    running = np.maximum.accumulate(np.maximum(path[:, 0], -path[:, 1]))
    excursion[i, : len(running)] = running

alive = (~np.isnan(excursion)).sum(axis=0)
x = np.arange(1, max_candles + 1)

fig = go.Figure()
for threshold, color in zip(TOUCH_THRESHOLDS, SERIES_HEX):
    touched = np.nansum(excursion >= threshold, axis=0) / np.maximum(alive, 1) * 100
    fig.add_trace(
        go.Scatter(
            x=x, y=touched, mode="lines", name=f"±{threshold:.0f}% touched",
            line=dict(color=color, width=2),
            hovertemplate="candle %{x} — %{y:.1f}% of weeks<extra></extra>",
        )
    )

for boundary in range(first_session, max_candles, CANDLES_PER_SESSION):
    fig.add_vline(x=boundary + 0.5, line_dash="dot", line_color="#c9c8c4", line_width=1)

fig.update_layout(
    title="Cumulative Share Of Weeks That Had Already Touched ±X% By Each Hourly Candle After Entry",
    template="plotly_white",
    height=460,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(t=80, b=50, l=60, r=20),
)
fig.update_xaxes(title_text=f"Hourly candles since the {ENTRY_TIME} entry (entry candle = 1)", gridcolor="#eceae6")
fig.update_yaxes(title_text="% of weeks touched", ticksuffix="%", gridcolor="#eceae6", rangemode="tozero")
fig.show()

n_sessions = int(moves["trading_days"].max())
session_range = range(1, n_sessions + 1)
timing = pd.DataFrame(
    {
        "Session since entry": [f"Session {d}" + (" (entry day)" if d == 1 else "") for d in session_range],
        "Weeks reaching this session": [int((moves["trading_days"] >= d).sum()) for d in session_range],
        "Weekly high printed here": (
            moves["session_of_max_up"].value_counts(normalize=True).mul(100)
            .reindex(session_range, fill_value=0.0).to_numpy()
        ),
        "Weekly low printed here": (
            moves["session_of_max_down"].value_counts(normalize=True).mul(100)
            .reindex(session_range, fill_value=0.0).to_numpy()
        ),
    }
)
display(timing.style.format({"Weekly high printed here": "{:.1f}%", "Weekly low printed here": "{:.1f}%"}).hide(axis="index"))

Session since entry,Weeks reaching this session,Weekly high printed here,Weekly low printed here
Session 1 (entry day),173,27.7%,27.7%
Session 2,173,19.1%,21.4%
Session 3,170,25.4%,30.1%
Session 4,128,26.6%,20.2%
Session 5,3,1.2%,0.6%
Session 6,1,0.0%,0.0%


## Where The Volatility Sits — Hour Of Day × Weekday

Mean hourly candle range, `(high − low) / open`, across every regular session in the dataset. One hue, light to dark: darker cells are the hours that move most. This is the intraday texture behind the weekly numbers above — useful for choosing an entry or adjustment hour rather than a date.

In [8]:
intraday = price_df.copy()
intraday["weekday"] = intraday.index.day_name()
intraday["hour"] = intraday.index.strftime("%H:%M")
intraday["range_pct"] = (intraday["high"] - intraday["low"]) / intraday["open"] * 100

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
heat = (
    intraday[intraday["weekday"].isin(weekday_order)]
    .pivot_table(index="weekday", columns="hour", values="range_pct", aggfunc="mean")
    .reindex(weekday_order)
)

fig = go.Figure(
    data=go.Heatmap(
        z=heat.to_numpy(),
        x=heat.columns,
        y=heat.index,
        colorscale=[[0, "#cde2fb"], [0.5, "#3987e5"], [1, "#0d366b"]],
        colorbar=dict(title="Mean range %", ticksuffix="%"),
        hovertemplate="%{y} %{x} — %{z:.2f}%<extra></extra>",
        xgap=2,
        ygap=2,
    )
)
fig.update_layout(
    title="Mean Hourly Candle Range By Weekday And Hour",
    template="plotly_white",
    height=380,
    margin=dict(t=60, b=50, l=90, r=20),
)
fig.update_xaxes(title_text="Candle start (IST)")
fig.show()

## Return Distribution — Entry-To-Expiry Close Move %

Every weekly `expiry_move_pct` bucketed into 1% return ranges, ordered best to worst, with the probability of landing at or beyond each bound — the same shape as a broker's option-probability table. Colour runs warm for gains, cool for losses, neutral through the middle.

In [9]:
edges = np.arange(LOWER_TAIL, UPPER_TAIL + BIN_WIDTH, BIN_WIDTH)
bin_edges = [-np.inf, *edges, np.inf]

labels = [f"< {LOWER_TAIL:.0f}%"]
labels += [f"{lo:.0f}% to {hi:.0f}%" for lo, hi in zip(edges[:-1], edges[1:])]
labels.append(f"> {UPPER_TAIL:.0f}%")

bucket = pd.cut(moves["expiry_move_pct"], bins=bin_edges, labels=labels, right=False)
dist = bucket.value_counts().reindex(labels[::-1]).rename("periods").to_frame()   # best bucket first

total = int(dist["periods"].sum())
dist["pct_of_total"] = dist["periods"] / total * 100
dist["prob_at_least"] = dist["periods"].cumsum() / total * 100          # P(return >= bucket lower bound)
dist["prob_at_most"] = 100 - dist["prob_at_least"] + dist["pct_of_total"]  # P(return <= bucket upper bound)


def diverging(n: int) -> list[str]:
    """Warm (best) to neutral to cool (worst), matching the up/down poles used throughout."""
    colors = []
    for t in np.linspace(1.0, -1.0, n):
        pole = UP_COLOR if t >= 0 else DOWN_COLOR
        mix = abs(t)
        colors.append("rgb({},{},{})".format(*[int(NEUTRAL[i] + (pole[i] - NEUTRAL[i]) * mix) for i in range(3)]))
    return colors


row_colors = diverging(len(dist))

fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[130, 110, 90, 140, 140],
            header=dict(
                values=[
                    "Return Range", "Number of Weeks", "% of Total",
                    "Probability (≥ lower bound)", "Probability (≤ upper bound)",
                ],
                fill_color=HEADER_FILL,
                font=dict(color="white", size=12),
                align="center",
                height=34,
            ),
            cells=dict(
                values=[
                    dist.index,
                    dist["periods"],
                    dist["pct_of_total"].map("{:.1f}%".format),
                    dist["prob_at_least"].map("{:.1f}%".format),
                    dist["prob_at_most"].map("{:.1f}%".format),
                ],
                fill_color=[row_colors] * 5,
                font=dict(color="black", size=11),
                align="center",
                height=28,
            ),
        )
    ]
)
fig.update_layout(
    title=f"Weekly Entry-To-Expiry Return Distribution ({total} weeks)",
    height=560,
    margin=dict(t=50, b=10, l=10, r=10),
)
fig.show()

## 🎯 Expiry Outliers & Global Events

Weeks whose entry-to-expiry return fell outside a chosen probability interval. Drag the slider to change the coverage: the counts, the scatter and the table all re-render together. Outliers are the weeks where a short strangle sized off the average would have been run over.

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import Markdown

RETURN_COL = "expiry_move_pct"
WITHIN_COLOR = "#8a8880"

outlier_base = moves.sort_values("curr_expiry").reset_index(drop=True)


def classify_outliers(coverage_pct: int):
    tail = (100 - coverage_pct) / 2
    lower_q, upper_q = tail, 100 - tail
    lower_bound = np.percentile(outlier_base[RETURN_COL], lower_q)
    upper_bound = np.percentile(outlier_base[RETURN_COL], upper_q)

    df = outlier_base.copy()
    df["status"] = np.select(
        [df[RETURN_COL] < lower_bound, df[RETURN_COL] > upper_bound],
        ["Downside outlier", "Upside outlier"],
        default="Within boundary",
    )
    return df, lower_bound, upper_bound, lower_q, upper_q


def render_outliers(coverage_pct=80):
    df, lower_bound, upper_bound, lower_q, upper_q = classify_outliers(coverage_pct)
    total = len(df)
    n_down = int((df["status"] == "Downside outlier").sum())
    n_up = int((df["status"] == "Upside outlier").sum())
    n_out = n_down + n_up

    display(Markdown(
        f"**{total} weeks** &nbsp;|&nbsp; at **{coverage_pct}%** coverage the boundaries are the "
        f"**{lower_q:.1f}th** percentile ({lower_bound:+.2f}%) and the **{upper_q:.1f}th** "
        f"percentile ({upper_bound:+.2f}%).  \n"
        f"Outliers: **{n_out}** ({n_out / total * 100:.1f}%) — **{n_down}** downside, **{n_up}** upside."
    ))

    color_map = {"Within boundary": WITHIN_COLOR, "Downside outlier": DOWN_HEX, "Upside outlier": UP_HEX}
    fig = go.Figure()
    for status, color in color_map.items():
        sub = df[df["status"] == status]
        fig.add_trace(
            go.Scatter(
                x=sub["curr_expiry"], y=sub[RETURN_COL], mode="markers", name=status,
                marker=dict(
                    color=color,
                    size=8 if status == "Within boundary" else 11,
                    line=dict(color="white", width=2),
                ),
                hovertemplate="%{x|%Y-%m-%d} — %{y:+.2f}%<extra></extra>",
            )
        )
    fig.add_hline(
        y=upper_bound, line_dash="dash", line_color=UP_HEX,
        annotation_text=f"Upper boundary ({upper_bound:+.2f}%)", annotation_position="top left",
    )
    fig.add_hline(
        y=lower_bound, line_dash="dash", line_color=DOWN_HEX,
        annotation_text=f"Lower boundary ({lower_bound:+.2f}%)", annotation_position="bottom left",
    )
    fig.update_layout(
        title="Weekly Expiry Returns — Inside And Outside The Chosen Boundary",
        template="plotly_white", height=460,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(t=80, b=50, l=60, r=20),
    )
    fig.update_xaxes(title_text="Expiry date", gridcolor="#eceae6")
    fig.update_yaxes(title_text="Entry-to-expiry return", ticksuffix="%", gridcolor="#eceae6")
    fig.show()

    out_df = df[df["status"] != "Within boundary"].copy()
    out_df = out_df.reindex(out_df[RETURN_COL].abs().sort_values(ascending=False).index)
    n_rows = len(out_df)
    status_colors = [DOWN_HEX if s == "Downside outlier" else UP_HEX for s in out_df["status"]]

    table_fig = go.Figure(
        data=[
            go.Table(
                columnwidth=[120, 90, 95, 95, 130],
                header=dict(
                    values=["Entry At", "Expiry", "Max Up %", "Max Down %", "Expiry Move %"],
                    fill_color=HEADER_FILL, font=dict(color="white", size=12), align="center", height=32,
                ),
                cells=dict(
                    values=[
                        out_df["entry_ts"].dt.strftime("%Y-%m-%d %H:%M"),
                        out_df["curr_expiry"].dt.strftime("%Y-%m-%d"),
                        out_df["max_up_pct"].map("{:+.2f}%".format),
                        out_df["max_down_pct"].map("{:+.2f}%".format),
                        out_df[RETURN_COL].map("{:+.2f}%".format),
                    ],
                    fill_color="white",
                    font=dict(color=[["black"] * n_rows] * 4 + [status_colors], size=11),
                    align="center", height=28,
                ),
            )
        ]
    )
    table_fig.update_layout(
        title="Outlier Weeks, Largest Absolute Move First",
        height=min(620, 90 + 28 * max(n_rows, 1)),
        margin=dict(t=50, b=10, l=10, r=10),
    )
    table_fig.show()


coverage_slider = widgets.IntSlider(
    value=80, min=50, max=99, step=1,
    description="Probability boundary coverage (%)",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="520px"),
)
interact(render_outliers, coverage_pct=coverage_slider)

interactive(children=(IntSlider(value=80, continuous_update=False, description='Probability boundary coverage …

<function __main__.render_outliers(coverage_pct=80)>